# IncidentIQ: IT Incident Management Analysis

**Skills shown:** ITSM incident analysis, SLA compliance, ticket routing, contact-channel analysis, recurring-incident (knowledge-base) analysis, Python / Pandas.

This notebook analyses an ITSM incident event log by priority, impact, assignment group and contact channel, and checks SLA compliance (`made_sla`). It then points to routing, staffing and knowledge-base improvements.

**Data:** place the incident event log CSV at `data/incident_event_log.csv` (for example the public *Incident management process enriched event log* dataset, 141,712 event records). The data is not included in this repository. Run the cells from top to bottom to produce every table and chart.

> **Records vs unique incidents:** an event log has several rows per incident (one per update). Section 2 reports SLA both ways, **per record** and **per unique incident**, so you can state exactly which one a headline figure refers to.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

os.makedirs('charts', exist_ok=True)
plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False, 'font.size': 10})
BLUE, GREEN, RED, AMBER, GREY = '#1F3A5F', '#4C9F70', '#C0392B', '#E0A030', '#8A94A0'

DATA_PATH = 'data/incident_event_log.csv'
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f'Put the incident event log CSV at {DATA_PATH} (see the README).')

raw = pd.read_csv(DATA_PATH, na_values=['?'], low_memory=False)
print('Rows:', len(raw), '| Columns:', raw.shape[1])
raw.head()

## 1. Clean the data

Column names are standardised, date columns are parsed (day first), and `made_sla` is converted to True / False.

In [ ]:
df = raw.copy()
df.columns = [c.strip().lower() for c in df.columns]

for c in ['opened_at', 'resolved_at', 'closed_at', 'sys_created_at', 'sys_updated_at']:
    if c in df.columns:
        df[c] = pd.to_datetime(df[c], dayfirst=True, errors='coerce')

df['made_sla'] = df['made_sla'].astype(str).str.strip().str.lower().map(
    {'true': True, 'false': False, '1': True, '0': False})
print('Missing made_sla values:', int(df['made_sla'].isna().sum()))

def clean_label(s):
    # '1 - Critical' -> '1 - Critical' (kept), but tidy whitespace
    return s.astype('string').str.strip()

for c in ['priority', 'impact', 'urgency', 'contact_type', 'category', 'subcategory', 'assignment_group']:
    if c in df.columns:
        df[c] = clean_label(df[c])

df[['number', 'incident_state', 'priority', 'impact', 'contact_type', 'assignment_group', 'made_sla']].head() if 'number' in df.columns else df.head()

## 2. Overall SLA compliance

In [ ]:
record_sla = df['made_sla'].mean() * 100
summary = {'Event records analysed': len(df), 'SLA met, per record (%)': round(record_sla, 1)}

if 'number' in df.columns:
    sort_cols = ['number'] + (['sys_updated_at'] if 'sys_updated_at' in df.columns else [])
    inc = df.sort_values(sort_cols).groupby('number', as_index=False).last()   # final state of each incident
    summary['Unique incidents'] = len(inc)
    summary['SLA met, per unique incident (%)'] = round(inc['made_sla'].mean() * 100, 1)
else:
    inc = df.copy()

pd.Series(summary, name='Value').to_frame()

## 3. SLA by priority, impact, contact channel and assignment group

Analysis is on every record (the same basis as the headline figure above). The `records` column is the volume behind each percentage.

In [ ]:
def sla_by(col, data=None, min_records=1):
    data = df if data is None else data
    g = (data.groupby(col, observed=True)
             .agg(records=('made_sla', 'size'), sla_met_pct=('made_sla', lambda s: s.mean() * 100))
             .sort_index())
    g = g[g['records'] >= min_records]
    g['sla_met_pct'] = g['sla_met_pct'].round(1)
    return g

by_priority = sla_by('priority')
by_priority

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.5))
bars = ax.bar(by_priority.index.astype(str), by_priority['sla_met_pct'], color=BLUE)
for b, v, n in zip(bars, by_priority['sla_met_pct'], by_priority['records']):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.8, f'{v:.1f}%\n(n={n:,})', ha='center', fontsize=9)
ax.axhline(record_sla, color=RED, ls='--', lw=1.3)
ax.text(len(by_priority) - 0.5, record_sla + 0.5, f'overall {record_sla:.1f}%', color=RED, ha='right', fontsize=9)
ax.set_ylim(0, 112)
ax.set_ylabel('SLA met (%)')
ax.set_xlabel('Priority')
ax.set_title('SLA compliance by priority', fontweight='bold')
plt.savefig('charts/sla_by_priority.png', dpi=130, bbox_inches='tight')
plt.show()

In [ ]:
by_impact = sla_by('impact')
by_channel = sla_by('contact_type').sort_values('records', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, tbl, title in [(axes[0], by_impact, 'SLA compliance by impact'),
                       (axes[1], by_channel, 'SLA compliance by contact channel')]:
    bars = ax.bar(tbl.index.astype(str), tbl['sla_met_pct'], color=BLUE)
    for b, v in zip(bars, tbl['sla_met_pct']):
        ax.text(b.get_x() + b.get_width() / 2, v + 0.8, f'{v:.1f}%', ha='center', fontsize=9)
    ax.set_ylim(0, 112)
    ax.set_ylabel('SLA met (%)')
    ax.set_title(title, fontweight='bold')
    ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.savefig('charts/sla_by_impact_and_channel.png', dpi=130, bbox_inches='tight')
plt.show()
by_channel

In [ ]:
by_group = sla_by('assignment_group', min_records=200).sort_values('records', ascending=False).head(10)
fig, ax = plt.subplots(figsize=(9, 5))
y = np.arange(len(by_group))[::-1]
colors = [RED if v < record_sla else BLUE for v in by_group['sla_met_pct']]
ax.barh(y, by_group['sla_met_pct'], color=colors)
for yi, v, n in zip(y, by_group['sla_met_pct'], by_group['records']):
    ax.text(v + 0.6, yi, f'{v:.1f}%  (n={n:,})', va='center', fontsize=9)
ax.set_yticks(y)
ax.set_yticklabels(by_group.index.astype(str))
ax.set_xlim(0, 118)
ax.set_xlabel('SLA met (%)')
ax.set_title('Top 10 assignment groups by volume (red = below overall rate)', fontweight='bold')
plt.savefig('charts/sla_by_assignment_group.png', dpi=130, bbox_inches='tight')
plt.show()
by_group

## 4. Routing: do reassignments hurt SLA?

Each time an incident is passed to another group it tends to take longer. This section checks whether incidents with more reassignments miss SLA more often.

In [ ]:
if 'reassignment_count' in df.columns:
    d = df.copy()
    d['reassignment_count'] = pd.to_numeric(d['reassignment_count'], errors='coerce')
    d['reassign_bucket'] = pd.cut(d['reassignment_count'], [-1, 0, 1, 2, 4, 1000],
                                  labels=['0', '1', '2', '3-4', '5+'])
    by_re = sla_by('reassign_bucket', data=d)
    fig, ax = plt.subplots(figsize=(7.5, 4.3))
    bars = ax.bar(by_re.index.astype(str), by_re['sla_met_pct'], color=BLUE)
    for b, v in zip(bars, by_re['sla_met_pct']):
        ax.text(b.get_x() + b.get_width() / 2, v + 0.8, f'{v:.1f}%', ha='center', fontsize=9)
    ax.set_ylim(0, 112)
    ax.set_xlabel('Reassignments on the record')
    ax.set_ylabel('SLA met (%)')
    ax.set_title('SLA compliance by number of reassignments', fontweight='bold')
    plt.savefig('charts/sla_by_reassignment.png', dpi=130, bbox_inches='tight')
    plt.show()
    by_re
else:
    print('reassignment_count column not found; skipping this section')

## 5. Resolution time (per unique incident)

In [ ]:
if 'number' in df.columns and {'opened_at', 'resolved_at'}.issubset(df.columns):
    t = df.groupby('number').agg(opened=('opened_at', 'min'), resolved=('resolved_at', 'max'),
                                 priority=('priority', 'last'))
    t['hours'] = (t['resolved'] - t['opened']).dt.total_seconds() / 3600
    t = t[(t['hours'] > 0) & t['priority'].notna()]
    cap = t['hours'].quantile(0.95)
    groups = sorted(t['priority'].dropna().unique())
    fig, ax = plt.subplots(figsize=(9, 4.8))
    ax.boxplot([t.loc[t['priority'] == g, 'hours'].clip(upper=cap) for g in groups],
               showfliers=False, patch_artist=True,
               boxprops=dict(facecolor='#DCE8F5', color=BLUE), medianprops=dict(color=RED))
    ax.set_xticks(range(1, len(groups) + 1))
    ax.set_xticklabels([str(g) for g in groups])
    ax.set_ylabel('Hours from opened to resolved (capped at 95th percentile)')
    ax.set_xlabel('Priority')
    ax.set_title('Resolution time by priority', fontweight='bold')
    plt.savefig('charts/resolution_time_by_priority.png', dpi=130, bbox_inches='tight')
    plt.show()
    t.groupby('priority')['hours'].describe()[['count', '50%', 'mean']].round(1).rename(columns={'50%': 'median_hours', 'mean': 'mean_hours'})
else:
    print('Needs number, opened_at and resolved_at columns; skipping')

## 6. Recurring incidents: knowledge-base candidates

Categories and subcategories that appear most often are the best candidates for knowledge-base articles, because one good article can deflect many repeat tickets.

In [ ]:
base = inc if 'number' in df.columns else df
cols = [c for c in ['category', 'subcategory'] if c in base.columns]
top_recurring = (base.groupby(cols, observed=True).size().sort_values(ascending=False).head(12).rename('incidents').reset_index())
top_recurring['label'] = top_recurring[cols].astype(str).agg(' / '.join, axis=1)

fig, ax = plt.subplots(figsize=(9, 5))
y = np.arange(len(top_recurring))[::-1]
ax.barh(y, top_recurring['incidents'], color=BLUE)
ax.set_yticks(y)
ax.set_yticklabels(top_recurring['label'], fontsize=9)
ax.set_xlabel('Incidents')
ax.set_title('Most frequent incident types (knowledge-base candidates)', fontweight='bold')
plt.savefig('charts/recurring_incident_types.png', dpi=130, bbox_inches='tight')
plt.show()
top_recurring[['label', 'incidents']]

In [ ]:
if 'knowledge' in df.columns:
    k = df.copy()
    k['knowledge'] = k['knowledge'].astype(str).str.strip().str.lower().map({'true': 'KB article used', 'false': 'No KB article'})
    kb = sla_by('knowledge', data=k)
    print('SLA compliance with and without a knowledge-base article:')
    print(kb)
kb if 'knowledge' in df.columns else None

## 7. Findings and recommendations

The text below is generated from the results above, so every figure matches the tables and charts.

In [ ]:
worst_group = by_group.sort_values('sla_met_pct').head(2)
print('FINDINGS')
print(f"1. Overall SLA compliance is {record_sla:.1f}% per record" +
      (f" and {summary['SLA met, per unique incident (%)']}% per unique incident." if 'Unique incidents' in summary else '.'))
print(f"2. Lowest SLA among the busiest groups: {worst_group.index[0]} ({worst_group['sla_met_pct'].iloc[0]}%) and {worst_group.index[1]} ({worst_group['sla_met_pct'].iloc[1]}%).")
print(f"3. Lowest SLA by priority: {by_priority['sla_met_pct'].idxmin()} ({by_priority['sla_met_pct'].min()}%).")
print(f"4. The most frequent incident type is {top_recurring['label'].iloc[0]} ({int(top_recurring['incidents'].iloc[0]):,} incidents).")
print()
print('RECOMMENDATIONS')
print(f"- Routing: review how incidents reach {worst_group.index[0]} and {worst_group.index[1]}, and cut avoidable reassignments.")
print(f"- Staffing: check coverage for {by_priority['sla_met_pct'].idxmin()} priority incidents at peak times.")
print(f"- Knowledge base: write or refresh articles for the top recurring incident types, starting with {top_recurring['label'].iloc[0]}.")